# Ex2 - Getting and Knowing your Data

This time we are going to pull data directly from the internet.
Special thanks to: https://github.com/justmarkham for sharing the dataset and materials.

### Step 1. Import the necessary libraries

In [1]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("ex2").getOrCreate()
spark

C:\Users\HEMANTH\anaconda3\lib\site-packages\pyspark\context.py:317: FutureWarning: Python 3.7 support is deprecated in Spark 3.4.
  warnings.warn("Python 3.7 support is deprecated in Spark 3.4.", FutureWarning)


### Step 2. Import the dataset from this [address](https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv). 

### Step 3. Assign it to a variable called chipo.

In [4]:
chipo = spark.read.csv('Chipotle.tsv', sep='\t', header = True, inferSchema=True)

### Step 4. See the first 10 entries

In [5]:
chipo.show(10)

+--------+--------+--------------------+--------------------+----------+
|order_id|quantity|           item_name|  choice_description|item_price|
+--------+--------+--------------------+--------------------+----------+
|       1|       1|Chips and Fresh T...|                NULL|    $2.39 |
|       1|       1|                Izze|        [Clementine]|    $3.39 |
|       1|       1|    Nantucket Nectar|             [Apple]|    $3.39 |
|       1|       1|Chips and Tomatil...|                NULL|    $2.39 |
|       2|       2|        Chicken Bowl|[Tomatillo-Red Ch...|   $16.98 |
|       3|       1|        Chicken Bowl|[Fresh Tomato Sal...|   $10.98 |
|       3|       1|       Side of Chips|                NULL|    $1.69 |
|       4|       1|       Steak Burrito|[Tomatillo Red Ch...|   $11.75 |
|       4|       1|    Steak Soft Tacos|[Tomatillo Green ...|    $9.25 |
|       5|       1|       Steak Burrito|[Fresh Tomato Sal...|    $9.25 |
+--------+--------+--------------------+-----------

### Step 5. What is the number of observations in the dataset?

In [6]:
# Solution 1
chipo.count()



4622

In [10]:
# Solution 2
len(chipo.columns)

5

### Step 6. What is the number of columns in the dataset?

In [12]:
len(chipo.columns)

5

### Step 7. Print the name of all the columns.

In [13]:
chipo.columns

['order_id', 'quantity', 'item_name', 'choice_description', 'item_price']

### Step 8. How is the dataset indexed?

### Step 9. Which was the most-ordered item? 

In [27]:
from pyspark.sql.functions import *
chipo.groupBy('item_name').agg(count('order_id').alias('total_orders')).orderBy('total_orders', ascending=False).show(5)

+-------------------+------------+
|          item_name|total_orders|
+-------------------+------------+
|       Chicken Bowl|         726|
|    Chicken Burrito|         553|
|Chips and Guacamole|         479|
|      Steak Burrito|         368|
|  Canned Soft Drink|         301|
+-------------------+------------+
only showing top 5 rows



### Step 10. For the most-ordered item, how many items were ordered?

In [25]:
chipo.groupBy('item_name').agg(sum('quantity').alias('quant')).orderBy('quant', ascending=False).show(5)

+-------------------+-----+
|          item_name|quant|
+-------------------+-----+
|       Chicken Bowl|  761|
|    Chicken Burrito|  591|
|Chips and Guacamole|  506|
|      Steak Burrito|  386|
|  Canned Soft Drink|  351|
+-------------------+-----+
only showing top 5 rows



### Step 11. What was the most ordered item in the choice_description column?

In [29]:
chipo.groupBy('choice_description').agg(count('order_id').alias('order_count')).orderBy('order_count',ascending=False).show()

+--------------------+-----------+
|  choice_description|order_count|
+--------------------+-----------+
|                NULL|       1246|
|         [Diet Coke]|        134|
|              [Coke]|        123|
|            [Sprite]|         77|
|[Fresh Tomato Sal...|         42|
|[Fresh Tomato Sal...|         40|
|[Fresh Tomato Sal...|         36|
|          [Lemonade]|         33|
|[Fresh Tomato Sal...|         33|
|[Fresh Tomato Sal...|         29|
|[Fresh Tomato Sal...|         28|
|[Fresh Tomato Sal...|         26|
|         [Coca Cola]|         26|
|[Fresh Tomato Sal...|         25|
|[Fresh Tomato Sal...|         23|
|[Fresh Tomato Sal...|         22|
|[Fresh Tomato Sal...|         22|
|[Fresh Tomato Sal...|         21|
|[Fresh Tomato Sal...|         21|
|[Roasted Chili Co...|         21|
+--------------------+-----------+
only showing top 20 rows



### Step 12. How many items were orderd in total?

In [59]:
chipo.select(sum('quantity')).show()

+-------------+
|sum(quantity)|
+-------------+
|         4972|
+-------------+



### Step 13. Turn the item price into a float

#### Step 13.a. Check the item price type

In [35]:
chipo.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- item_name: string (nullable = true)
 |-- choice_description: string (nullable = true)
 |-- item_price: string (nullable = true)



#### Step 13.b. Create a lambda function and change the type of item price

In [37]:
chipo = chipo.withColumn(
    'item_price',
    regexp_replace(col('item_price'), "[$]","").cast('float')
)

In [39]:
chipo.show(5)

+--------+--------+--------------------+--------------------+----------+
|order_id|quantity|           item_name|  choice_description|item_price|
+--------+--------+--------------------+--------------------+----------+
|       1|       1|Chips and Fresh T...|                NULL|      2.39|
|       1|       1|                Izze|        [Clementine]|      3.39|
|       1|       1|    Nantucket Nectar|             [Apple]|      3.39|
|       1|       1|Chips and Tomatil...|                NULL|      2.39|
|       2|       2|        Chicken Bowl|[Tomatillo-Red Ch...|     16.98|
+--------+--------+--------------------+--------------------+----------+
only showing top 5 rows



#### Step 13.c. Check the item price type

In [40]:
chipo.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- item_name: string (nullable = true)
 |-- choice_description: string (nullable = true)
 |-- item_price: float (nullable = true)



### Step 14. How much was the revenue for the period in the dataset?

In [46]:
chipo = chipo.withColumn('Revenue', col('quantity') * col('item_price'))
chipo.select(sum('Revenue')).show()

+----------------+
|    sum(Revenue)|
+----------------+
|39237.0197327137|
+----------------+



In [47]:
total = chipo.agg(sum('Revenue')).collect()[0][0]
print(total)

39237.0197327137


### Step 15. How many orders were made in the period?

In [45]:
chipo.select('order_id').distinct().count()

1834

### Step 16. What is the average revenue amount per order?

In [53]:
# Solution 1

chipo.groupBy('order_id') \
     .agg(round(avg('Revenue'),2).alias('Avg per order')) \
     .orderBy('Avg per order', ascending=False).show()

+--------+-------------+
|order_id|Avg per order|
+--------+-------------+
|    1443|       153.46|
|    1398|       105.75|
|     178|        98.82|
|     616|        78.75|
|    1336|        78.21|
|     193|         66.6|
|    1559|         49.2|
|     253|        47.56|
|     123|        47.56|
|    1502|         47.0|
|    1175|        45.92|
|     511|        45.04|
|     261|         45.0|
|     357|         45.0|
|      94|         45.0|
|     578|         45.0|
|    1409|         45.0|
|     459|         45.0|
|      60|         45.0|
|     777|         45.0|
+--------+-------------+
only showing top 20 rows



In [4]:
# Solution 2



### Step 17. How many different items are sold?

In [56]:
chipo.select('item_name').distinct().count()

50